        # 01 年齢別所得と進学の比較（RQ1）

        **F1：キャリア選択と生涯所得の最適化**

        - 問い：**18 歳・20 歳・22 歳で就職すると、生涯所得はどう変わるか**
        - 方法：年齢別の年収をリスト（配列）で計算し、ループで累積する

> 架空のデータによる**教育用シミュレーション**です。実際の賃金統計ではなく、将来の所得の予測でもありません。金額は**万円**、税・社会保険料を引く前の**額面**です。

## 準備：実験環境（最初に必ず実行）

F1 リポジトリの `.py`（`scenarios.py`・`income_analysis.py`・`optimization.py` など）を import できるように、作業ディレクトリ（F1 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | F1 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

> `textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと `optimization.py` の import が失敗します。

In [ ]:
#@title 準備（F1 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                                  # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "f1-career-income-optimization"  # リポジトリの中の F1 フォルダ（ルートが F1 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_f1_root(p):
    return (Path(p) / "career_model.py").exists() and (Path(p) / "optimization.py").exists()


def find_f1_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または F1 のルート
        if is_f1_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_f1_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("F1 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/f1_upload")
    for hit in glob.glob("/content/f1_upload/**/career_model.py", recursive=True):
        if is_f1_root(Path(hit).parent):
            return Path(hit).parent
    return None


F1_ROOT = find_f1_root()
if F1_ROOT is None:
    raise RuntimeError("F1 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(F1_ROOT)
if str(F1_ROOT) not in sys.path:
    sys.path.insert(0, str(F1_ROOT))
if not (F1_ROOT / "textbook" / "max.py").exists():
    print("⚠️ textbook/ がありません。optimization.py の import が失敗します。")

# ---- 既存の .py（計算の本体）と、発表用のグラフ ----
import matplotlib.pyplot as plt
import career_model as cm
import scenarios as sc
import income_analysis as ia
import optimization as op
import presentation_charts as pc

pc.setup_style()                      # 発表用に文字を大きめにする（pc.setup_style(1.3) でさらに大きく）
careers, settings = sc.load_scenarios()
named = {s["name"]: s for s in careers}
RATE = settings.get("discount_rate", ia.DEFAULT_DISCOUNT_RATE)

SAVE_FIGURES = False                  # True にすると、スライド用の PNG を notebooks/figures/ に保存する
FIG_DIR = "notebooks/figures"

def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", pc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()

print("F1_ROOT:", F1_ROOT)
print("キャリア:", [s["label"] for s in careers], " 割引率:", RATE)

## モデル：年齢別の年収

$$ \text{salary}(a) = \text{initial\_salary} \times (1 + g)^{a - \text{start\_age}} \qquad (\text{start\_age} \le a < 65) $$

- 就職前（$a < \text{start\_age}$）は 0
- 65 歳の年からは計算しない（**18〜64 歳の 47 年間**）
- 年齢別の値は**長さ 47 のリスト**：i 番目 = (18 + i) 歳

In [ ]:
# 比べる 3 つのキャリアの仮定（data/sample_careers.json）
rq1 = [named["early_job"], named["vocational"], named["university"]]
for s in rq1:
    print(f"{s['label']:16} 就職 {s['start_age']} 歳  初年度 {s['initial_salary']} 万円  "
          f"昇給率 {s['annual_growth_rate']:.1%}  進学費用 {s['education_cost']} 万円")

## 年齢別の年収

進学すると、就職までの**所得は 0**（機会費用）。その代わり、この架空データでは初年度年収・昇給率を高く仮定しています。

In [ ]:
#@title グラフを表示（コードは隠しています）
show(pc.plot_income_by_age(rq1, "RQ1: annual income by age"), "rq1_income_by_age")

## 累積純収入：いつ追いつくか

累積純収入 = 年収 − 進学費用 を、18 歳から**累積和**したもの。点は 18 歳就職（Early Job）に追いつく年齢（**投資回収年齢**）。

In [ ]:
#@title グラフを表示（コードは隠しています）
show(pc.plot_cumulative_net(rq1, baseline=named["early_job"], title="RQ1: cumulative net income"),
     "rq1_cumulative_net")

In [ ]:
for s in rq1[1:]:
    p = ia.payback_age(s, named["early_job"])
    print(f"{s['label']:16} → Early Job に追いつく年齢：{p['payback_age']} 歳（状態：{p['status']}）、"
          f"64 歳時点の差 {p['final_gap']:+,.0f} 万円")

## 結果の表

In [ ]:
results = [ia.evaluate(s, RATE) for s in rq1]
print(f"{'Scenario':16}{'Lifetime Income':>17}{'Costs':>8}{'Net Income':>12}{'NPV':>10}")
for r in results:
    print(f"{r['label']:16}{r['gross']:>17,.0f}{r['costs']:>8,.0f}{r['net']:>12,.0f}{r['npv']:>10,.0f}")
print(f"（万円、NPV は割引率 {RATE:.0%}）")
show(pc.plot_metric_bars(results, "net", "RQ1: cumulative net income (lifetime income - costs)"),
     "rq1_net_bars")

## 仮定を変えてみる（感度）

`with_changes` は、**元のシナリオを変えずに**コピーの値だけを変えます。下の値を変えて、もう一度実行してください。

In [ ]:
# TODO：値を変えて再実行しよう
uni = sc.with_changes(named["university"], initial_salary=300, annual_growth_rate=0.02, education_cost=400)

p = ia.payback_age(uni, named["early_job"])
print("変更後の University：回収年齢", p["payback_age"], "/ 状態", p["status"],
      f"/ 64 歳時点の差 {p['final_gap']:+,.0f} 万円")
show(pc.plot_gap_vs_baseline(uni, named["early_job"]), "rq1_gap_modified")

## アルゴリズムと計算量

| 処理 | 方法 | 計算量 |
| --- | --- | --- |
| 年齢別の年収 | 年齢ごとに式で計算（リストに入れる） | $O(T)$、$T = 47$ |
| 累積純収入 | ループで足していく（累積和） | $O(T)$ |
| 投資回収年齢 | 差が 0 以上になる最初の年齢を**先頭から線形探索** | $O(T)$ |
| $N$ 個のキャリア | それぞれ計算 | $O(NT)$ |

差は単調に増えるとは限らないので、二分探索ではなく線形探索を使います。

## 考察（チームで記入）

- **TODO**：18・20・22 歳就職の順位と、その**理由**（どの仮定が効いているか）
- **TODO**：「進学すると年収・昇給率が上がる」という仮定は、どこまで正しいと言えるか（相関と因果）
- **TODO**：このモデルで**考えていないこと**（税・手取り、学費以外の費用、奨学金、学ぶこと自体の価値 など）

## 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第2章 最大値、第3章 線形探索、第6章 挿入ソート）
- F1 リポジトリの `README.md`（モデルの仮定、計算量、モデルの限界と倫理的配慮）
- データ：`data/sample_careers.json`（この教材のための**架空のデータ**）